# Test Image Creation

This notebook is a development environment of a pipeline to create and save images for LFD detector testing.

## Notes for Pipeline
- Should keep functions to a minimum, unless anticipated to be generalized
- Current implementation uses units of electrons. However the true images are difference images from the LSST Camera that may not have the same unit. The code should be modified for this change.
- Output format remains to be decided.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.special import erf

# Constants
NX, NY = (4096, 4004)
READ_NOISE = 7.0  # in electrons per pixel
FWHM = 0.7  # in arcseconds
PIXEL_SCALE = 0.2  # in arcseconds per pixel
SIGMA = (FWHM / 0.2) / (2 * math.sqrt(2 * math.log(2)))
SKY_COUNTS = {
    "u": 81.0,
    "g": 411.0,
    "r": 819.0,
    "i": 1173.0,
    "z": 1784.0,
    "Y": 2371.0,
}  # in electrons per pixel

# Inputs/Configs
band = "i"  # filter band name
theta = 0.0  # in degrees (Hesse Normal)
rho = 2000.0  # in pixels (Hesse Normal)
width = 20.0  # in pixels
peak_signal = 1000.0  # in electrons

## Select a streak profile

First-order: Model the cross-section profile of the streak as a Top-Hat convolved with a Gaussian with fwhm = 0.7". Calculate the signal in each pixel of the image array using its distance from the line. 

In [2]:
def blurred_box(x, width, sigma):
    y = 0.5 * (
        erf((x + width/2)/(np.sqrt(2)*sigma))
        - erf((x - width/2)/(np.sqrt(2)*sigma))
    )

    # Normalize so max = 1
    y /= erf(width/(2*np.sqrt(2)*sigma))

    return y

# Calculate the signal in each pixel (reshape a simplification?)
gy, gx = np.ogrid[:NY, :NX]
distance = gx*np.cos(theta) + gy*np.sin(theta) - rho
streak_signal = blurred_box(distance.flatten(), width, SIGMA) * peak_signal

## Create the necessary image planes

Create image plane from read noise, sky background, and streak signal. Background is then subtracted. Create corresponding variance plane. For now create an empty bad mask plane.

In [3]:
# Create image plane
rng = np.random.default_rng()
image = rng.normal(loc=0.0, scale=READ_NOISE, size=(NY, NX))
image += rng.poisson(lam=SKY_COUNTS[band], size=(NY, NX)) - SKY_COUNTS[band]
image += rng.poisson(streak_signal, NY * NX).reshape((NY, NX))

# Create variance plane
variance = np.full((NY, NX), READ_NOISE**2)
variance += SKY_COUNTS[band]
variance += streak_signal.reshape((NY, NX))

# Create mask plane
mask = np.zeros((NY, NX), dtype=bool)

## Output

2 goals:
- save as a format that is optimal for reducing disk storage use
- save as a format that is compatabile with `lsst.afw.image.Exposure` for eventual LSST Science Pipelines testing.

If there is no good candidate, pick the best for each case and provide a plug-and-play solution, as well as tools for quick conversions.